# YOLO11 Model Kullanımı / YOLO11 Model Usage

Bu notebook, YOLO11 segmentasyon modelinin nasıl yükleneceğini ve kullanılacağını gösterir.
This notebook demonstrates how to load and use the YOLO11 segmentation model for defect detection and ore classification.

## Turkish Class Names (utils.py referansı):
- **manyetit** (magnetite) - Red
- **krom** (chromite) - Green
- **atık** (waste) - Gray
- **düşük tenör** (low grade) - Orange
- **defect** (defect) - Magenta
- **normal** - Light green

In [ ]:
# Import required libraries
import os
import sys
from pathlib import Path

# Add project root to path
project_root = Path('../').resolve()
sys.path.insert(0, str(project_root))

import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# Import from utils
from utils import get_ore_class_colors, get_defect_colors, draw_annotations

## Model Yükleme / Loading Model

In [ ]:
# YOLO11 Model Import
from ultralytics import YOLO

# Model seçenekleri / Model options:
# - yolo11n-seg.pt (nano - en hızlı, en az doğru)
# - yolo11s-seg.pt (small)
# - yolo11m-seg.pt (medium)
# - yolo11l-seg.pt (large)
# - yolo11x-seg.pt (xlarge - en yavaş, en doğru)

MODEL_NAME = "yolo11n-seg.pt"  # Varsayılan olarak nano model

print(f"Loading YOLO11 model: {MODEL_NAME}")
model = YOLO(MODEL_NAME)
print("Model loaded successfully!")

# Model bilgilerini göster / Display model info
print(f"Model classes: {model.names}")

## Turkish Class Names Tanımlama / Defining Turkish Class Names

In [ ]:
# Turkish class names for ore classification (from utils.py)
TURKISH_CLASS_NAMES = {
    0: 'manyetit',    # Magnetite
    1: 'krom',        # Chromite
    2: 'atık',        # Waste
    3: 'düşük tenör', # Low grade
    4: 'normal'       # Normal
}

# Turkish defect class names
TURKISH_DEFECT_NAMES = {
    0: 'çatlak',      # Crack
    1: 'çizik',       # Scratch
    2: 'delik',       # Hole
    3: 'leke',        # Stain
    4: 'deformasyon'  # Deformation
}

# Renkler / Colors (from utils.py)
ore_colors = get_ore_class_colors()
defect_colors = get_defect_colors()

print("Turkish Ore Classes:", TURKISH_CLASS_NAMES)
print("Turkish Defect Classes:", TURKISH_DEFECT_NAMES)
print("Ore Colors:", ore_colors)
print("Defect Colors:", defect_colors)

## Örnek Görüntü ile Çıkarım / Inference on Sample Image

In [ ]:
# Sample image path - use a placeholder or create test image
# Gerçek görüntü yolu / Real image path:
SAMPLE_IMAGE_PATH = "../datasets/defect_detection/train/images/sample.jpg"

# Görüntüyü yükle veya örnek oluştur / Load image or create sample
if os.path.exists(SAMPLE_IMAGE_PATH):
    image = cv2.imread(SAMPLE_IMAGE_PATH)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
else:
    # Create a sample test image if not exists
    print("Sample image not found. Creating a test image...")
    # Create a 640x640 test image
    image = np.random.randint(0, 255, (640, 640, 3), dtype=np.uint8)
    image_rgb = image.copy()

print(f"Image shape: {image_rgb.shape}")

# Display the image
plt.figure(figsize=(10, 10))
plt.imshow(image_rgb)
plt.title("Input Image / Giriş Görüntüsü")
plt.axis('off')
plt.show()

## Model Çıkarımı / Model Inference

In [ ]:
# Inference parameters
CONF_THRESHOLD = 0.25  # Güven eşiği / Confidence threshold
IOU_THRESHOLD = 0.45   # IoU eşiği / IoU threshold
IMG_SIZE = 640         # Görüntü boyutu / Image size

# Run inference
results = model.predict(
    image,
    conf=CONF_THRESHOLD,
    iou=IOU_THRESHOLD,
    imgsz=IMG_SIZE,
    verbose=False
)

print(f"Number of detections: {len(results[0].boxes) if results else 0}")

# Display results
if results and len(results[0].boxes) > 0:
    boxes = results[0].boxes
    print("\nDetections / Algılamalar:")
    for i, box in enumerate(boxes):
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])
        # Use Turkish class name if available
        class_name = TURKISH_CLASS_NAMES.get(cls_id, f"class_{cls_id}")
        print(f"  {i+1}. {class_name}: {conf:.2%}")
else:
    print("No detections found.")

## Sonuçları Görselleştirme / Visualizing Results

In [ ]:
# Use the built-in plot method
fig, axes = plt.subplots(1, 2, figsize=(15, 7))

# Original image
axes[0].imshow(image_rgb)
axes[0].set_title("Orijinal Görüntü / Original Image")
axes[0].axis('off')

# Results with boxes
if results:
    # Plot with results
    result_plot = results[0].plot()  # Returns numpy array
    axes[1].imshow(result_plot)
    axes[1].set_title("Algılama Sonuçları / Detection Results")
else:
    axes[1].imshow(image_rgb)
    axes[1].set_title("Sonuç Yok / No Results")

axes[1].axis('off')

plt.tight_layout()
plt.show()

## Özel Görselleştirme (utils.py ile) / Custom Visualization with utils.py

In [ ]:
# Custom visualization using utils.py functions
annotated_image = draw_annotations(
    image,
    results,
    class_colors=ore_colors,
    show_labels=True,
    show_confidence=True,
    line_thickness=2
)

# Display
plt.figure(figsize=(10, 10))
plt.imshow(annotated_image)
plt.title("Özel Görselleştirme / Custom Visualization")
plt.axis('off')
plt.show()

# Save the annotated image
output_path = "../outputs/detection_result.jpg"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
annotated_image_bgr = cv2.cvtColor(annotated_image, cv2.COLOR_RGB2BGR)
cv2.imwrite(output_path, annotated_image_bgr)
print(f"Annotated image saved to: {output_path}")

## Toplu İşlem / Batch Processing

In [ ]:
# Batch inference on multiple images
def process_images(model, image_paths, conf=0.25):
    """
    Process multiple images with YOLO model.
    
    Args:
        model: YOLO model
        image_paths: List of image paths
        conf: Confidence threshold
    
    Returns:
        List of results
    """
    results = []
    for path in image_paths:
        if os.path.exists(path):
            result = model.predict(path, conf=conf, verbose=False)[0]
            results.append({'path': path, 'result': result})
    return results

# Example: Process a folder of images
image_folder = "../datasets/defect_detection/train/images"
if os.path.exists(image_folder):
    image_files = [os.path.join(image_folder, f) for f in os.listdir(image_folder) 
                   if f.endswith(('.jpg', '.jpeg', '.png'))][:5]  # Limit to 5
    
    if image_files:
        print(f"Processing {len(image_files)} images...")
        batch_results = process_images(model, image_files)
        
        for res in batch_results:
            num_dets = len(res['result'].boxes) if res['result'] else 0
            print(f"{os.path.basename(res['path'])}: {num_dets} detections")
else:
    print(f"Image folder not found: {image_folder}")

## Model Bilgisi ve Performans / Model Info and Performance

In [ ]:
# Model information
print("=" * 50)
print("MODEL BİLGİSİ / MODEL INFORMATION")
print("=" * 50)
print(f"Model name: {MODEL_NAME}")
print(f"Task: {model.task}")
print(f"Classes: {model.names}")

# Model parameters
if hasattr(model, 'model'):
    total_params = sum(p.numel() for p in model.model.parameters())
    trainable_params = sum(p.numel() for p in model.model.parameters() if p.requires_grad)
    print(f"\nTotal parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")

# Benchmark inference time
import time

iterations = 100
start_time = time.time()
for _ in range(iterations):
    _ = model.predict(image, verbose=False)
end_time = time.time()

avg_time = (end_time - start_time) / iterations * 1000
print(f"\nAverage inference time: {avg_time:.2f} ms")
print(f"FPS: {1000/avg_time:.1f}")

## Özet / Summary

Bu notebook şunları gösterdi:
- YOLO11 modelinin yüklenmesi
- Türkçe sınıf isimlerinin kullanımı (manyetit, krom, atık, düşük tenör, normal)
- Tek görüntü üzerinde çıkarım
- Sonuçların görselleştirilmesi
- utils.py fonksiyonlarının kullanımı
---
This notebook demonstrated:
- Loading YOLO11 model
- Using Turkish class names (manyetit, krom, atık, düşük tenör, normal)
- Running inference on a single image
- Visualizing results
- Using utils.py functions